In [ ]:
#Figure 4b -- Isohydricity

from pathlib import Path
import warnings

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import xarray as xr
from scipy.io import loadmat
from scipy.stats import gaussian_kde

BASE = Path(r"D:/Research/Objective2/Isohydricity/Beta_with_IF")

mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.weight": "bold",
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "text.color": "black",
    "axes.labelcolor": "black",
    "axes.edgecolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
})

REGIONS = ["Amazon", "China", "Congo", "Australia", "Taiga"]
REGION_TITLES = {
    "Amazon": "Amazon Basin", "Congo": "Congo Basin",
    "Australia": "Eastern Australia", "China": "Southeast China",
    "Taiga": "Taiga-Russia",
}

bivar = {
    "Amazon": {"beta": BASE/"beta_Amazon_025.nc", "sm_lai": BASE/"Amazon_b_sm_lai_mean_map.mat",
               "vpd_lai": BASE/"Amazon_b_vpd_lai_mean_map.mat", "sm_gpp": BASE/"Amazon_b_sm_gpp_mean_map.mat",
               "vpd_gpp": BASE/"Amazon_b_vpd_gpp_mean_map.mat"},
    "China": {"beta": BASE/"beta_China_025.nc", "sm_lai": BASE/"china_b_sm_lai_mean_map.mat",
              "vpd_lai": BASE/"china_b_vpd_lai_mean_map.mat", "sm_gpp": BASE/"china_b_sm_gpp_mean_map.mat",
              "vpd_gpp": BASE/"china_b_vpd_gpp_mean_map.mat"},
    "Congo": {"beta": BASE/"beta_Congo_025.nc", "sm_lai": BASE/"Congo_b_sm_lai_mean_map.mat",
              "vpd_lai": BASE/"Congo_b_vpd_lai_mean_map.mat", "sm_gpp": BASE/"Congo_b_sm_gpp_mean_map.mat",
              "vpd_gpp": BASE/"Congo_b_vpd_gpp_mean_map.mat"},
    "Australia": {"beta": BASE/"beta_Australia_025.nc", "sm_lai": BASE/"Australia_b_sm_lai_mean_map.mat",
                  "vpd_lai": BASE/"Australia_b_vpd_lai_mean_map.mat", "sm_gpp": BASE/"Australia_b_sm_gpp_mean_map.mat",
                  "vpd_gpp": BASE/"Australia_b_vpd_gpp_mean_map.mat"},
    "Taiga": {"beta": BASE/"beta_Taiga_025.nc", "sm_lai": BASE/"Taiga_b_sm_lai_mean_map.mat",
              "vpd_lai": BASE/"Taiga_b_vpd_lai_mean_map.mat", "sm_gpp": BASE/"Taiga_b_sm_gpp_mean_map.mat",
              "vpd_gpp": BASE/"Taiga_b_vpd_gpp_mean_map.mat"},
}

multivar = {
    "Amazon": {"beta": BASE/"beta_Amazon_025.nc", "sm_lai": BASE/"Amazon_m_sm_lai_vpd_t_ssr_mean_map.mat",
               "vpd_lai": BASE/"Amazon_m_vpd_lai_sm_t_ssr_mean_map.mat", "sm_gpp": BASE/"Amazon_m_sm_gpp_vpd_t_ssr_mean_map.mat",
               "vpd_gpp": BASE/"Amazon_m_vpd_gpp_sm_t_ssr_mean_map.mat"},
    "China": {"beta": BASE/"beta_China_025.nc", "sm_lai": BASE/"china_m_sm_lai_vpd_t_ssr_mean_map.mat",
              "vpd_lai": BASE/"china_m_vpd_lai_sm_t_ssr_mean_map.mat", "sm_gpp": BASE/"china_m_sm_gpp_vpd_t_ssr_mean_map.mat",
              "vpd_gpp": BASE/"china_m_vpd_gpp_sm_t_ssr_mean_map.mat"},
    "Congo": {"beta": BASE/"beta_Congo_025.nc", "sm_lai": BASE/"Congo_m_sm_lai_vpd_t_ssr_mean_map.mat",
              "vpd_lai": BASE/"Congo_m_vpd_lai_sm_t_ssr_mean_map.mat", "sm_gpp": BASE/"Congo_m_sm_gpp_vpd_t_ssr_mean_map.mat",
              "vpd_gpp": BASE/"Congo_m_vpd_gpp_sm_t_ssr_mean_map.mat"},
    "Australia": {"beta": BASE/"beta_Australia_025.nc", "sm_lai": BASE/"Australia_m_sm_lai_vpd_t_ssr_mean_map.mat",
                  "vpd_lai": BASE/"Australia_m_vpd_lai_sm_t_ssr_mean_map.mat", "sm_gpp": BASE/"Australia_m_sm_gpp_vpd_t_ssr_mean_map.mat",
                  "vpd_gpp": BASE/"Australia_m_vpd_gpp_sm_t_ssr_mean_map.mat"},
    "Taiga": {"beta": BASE/"beta_Taiga_025.nc", "sm_lai": BASE/"Taiga_m_sm_lai_vpd_t_ssr_mean_map.mat",
              "vpd_lai": BASE/"Taiga_m_vpd_lai_sm_t_ssr_mean_map.mat", "sm_gpp": BASE/"Taiga_m_sm_gpp_vpd_t_ssr_mean_map.mat",
              "vpd_gpp": BASE/"Taiga_m_vpd_gpp_sm_t_ssr_mean_map.mat"},
}

SM_LAI = "#603c87"
SM_GPP = "#0072b2"
VPD_LAI = "#ffbd15"
VPD_GPP = "#d35f00"
CENTER_COLOR = "#808080"


def load_beta_and_weights(path):
    """Read beta and aligned cosine(latitude) on the *beta file's* grid."""
    with xr.open_dataset(path) as ds:
        var = "beta" if "beta" in ds.data_vars else list(ds.data_vars)[0]
        beta_da = ds[var].squeeze(drop=True)
        if beta_da.ndim != 2:
            raise ValueError(f"{path.name}: expected a 2-D spatial beta map; got {beta_da.dims}: {beta_da.shape}")

        lat_name = next((name for name in ("lat", "latitude", "Latitude", "LAT")
                         if name in ds.variables), None)
        if lat_name is None:
            raise ValueError(f"{path.name}: no latitude coordinate named lat or latitude")
        lat_da = ds[lat_name].squeeze(drop=True)

       
        _, lat_broadcast = xr.broadcast(beta_da, lat_da)
        if set(lat_broadcast.dims) != set(beta_da.dims):
            raise ValueError(f"{path.name}: latitude has incompatible dims: {lat_da.dims} vs {beta_da.dims}")
        lat_broadcast = lat_broadcast.transpose(*beta_da.dims)

        beta_arr = np.asarray(beta_da.values, dtype=float)
        latitude = np.asarray(lat_broadcast.values, dtype=float)
        if latitude.shape != beta_arr.shape:
            raise ValueError(f"{path.name}: beta/latitude grids differ")
        weights = np.cos(np.deg2rad(latitude))
        if np.any((weights <= 0) & np.isfinite(weights)):
            raise ValueError(f"{path.name}: invalid cosine(latitude) weights")
        print(f"  Beta dims: {beta_da.dims}, shape: {beta_da.shape}, "
              f"latitude range: [{np.nanmin(latitude):.2f}, {np.nanmax(latitude):.2f}]")
        return beta_arr.ravel(order="C"), weights.ravel(order="C"), beta_arr.shape


def load_mat_main_array(path, expected_shape):
    md = loadmat(path)
    cands = [(k, v) for k, v in md.items()
             if not k.startswith("__") and isinstance(v, np.ndarray)
             and np.issubdtype(v.dtype, np.number)]
    if not cands:
        raise ValueError(f"{path.name}: no numeric arrays found")
    name, raw = max(cands, key=lambda item: item[1].size)
    arr = np.asarray(np.squeeze(raw), dtype=float)

    if arr.size != int(np.prod(expected_shape)):
        raise ValueError(f"{path.name}: {arr.size} cells; beta has {np.prod(expected_shape)}")
    if arr.ndim == 2 and arr.shape != expected_shape:
        raise ValueError(
            f"{path.name}: IF shape {arr.shape} differs from beta shape {expected_shape}; "
            "check geographic ordering rather than silently transposing."
        )
    if arr.ndim == 1:
        # Flat vector: checking cell counts cannot independently prove spatial order.
        warnings.warn(f"{path.name}: IF map is flat. Verify its pixel sequence matches the beta map.")
    return arr.ravel(order="C")


def weighted_quantiles(values, weights, quantiles):
    """Weighted empirical (inverse-CDF) quantiles, valid on old NumPy too."""
    x = np.asarray(values, dtype=float)
    w = np.asarray(weights, dtype=float)
    q = np.atleast_1d(quantiles)
    mask = np.isfinite(x) & np.isfinite(w) & (w > 0)
    x, w = x[mask], w[mask]
    if len(x) == 0:
        return np.full(q.shape, np.nan)
    order = np.argsort(x, kind="stable")
    x, w = x[order], w[order]
    cdf = np.cumsum(w) / np.sum(w)
    ix = np.searchsorted(cdf, q, side="left")
    return x[np.clip(ix, 0, len(x)-1)]


def prepare_payload(entry):
    beta, weights, shape = load_beta_and_weights(entry["beta"])
    arrays = {key: load_mat_main_array(entry[key], shape)
              for key in ("sm_lai", "vpd_lai", "sm_gpp", "vpd_gpp")}
    mask = np.isfinite(beta) & np.isfinite(weights) & (weights > 0)
    for arr in arrays.values():
        mask &= np.isfinite(arr)
    if np.count_nonzero(mask) < 3:
        raise ValueError(f"{entry['beta']}: fewer than 3 valid co-located pixels")

    beta = beta[mask]
    weights = weights[mask]
    arrays = {k: v[mask] for k, v in arrays.items()}

    
    q1, q2 = np.nanpercentile(beta, [33.333, 66.667])
    bins = {
        "Low β (Iso)": beta <= q1,
        "Mid β": (beta > q1) & (beta <= q2),
        "High β (Aniso)": beta > q2,
    }
    print(f"  Valid pixels: {len(beta)}; unweighted β terciles: {q1:.4g}, {q2:.4g}")
    for label, m in bins.items():
        print(f"    {label}: {np.count_nonzero(m)} pixels, "
              f"{100 * np.sum(weights[m]) / np.sum(weights):.1f}% of represented area")
    return {"beta": beta, "weights": weights, "bins": bins, **arrays}


def groups(values, weights, bins):
    order = ("Low β (Iso)", "Mid β", "High β (Aniso)")
    return [values[bins[k]] for k in order], [weights[bins[k]] for k in order]


def kde_density(values, weights, y):
    good = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    values, weights = values[good], weights[good]
    if values.size < 2 or np.ptp(values) <= 1e-14:
        return np.zeros_like(y)
    try:
        return gaussian_kde(values, weights=weights)(y)
    except np.linalg.LinAlgError:
        warnings.warn("KDE degenerate for a beta bin; omitting its density")
        return np.zeros_like(y)


def split_violin(ax, left_groups, right_groups, left_weights, right_weights,
                 left_color, right_color, centers=(1, 2, 3)):
    all_finite = [g[np.isfinite(g)] for g in left_groups + right_groups]
    all_finite = [g for g in all_finite if len(g)]
    if not all_finite:
        return
    all_vals = np.concatenate(all_finite)
   
    
    y_min, y_max = np.nanpercentile(all_vals, [1, 99])
    pad = 0.05*(y_max-y_min + 1e-12)
    y = np.linspace(y_min-pad, y_max+pad, 200)

    for cx, lv, rv, lw, rw in zip(centers, left_groups, right_groups,
                                  left_weights, right_weights):
        dl = kde_density(lv, lw, y)
        dr = kde_density(rv, rw, y)
        mx = max(float(dl.max()), float(dr.max()), 1e-12)
        wl = (dl/mx)*0.35
        wr = (dr/mx)*0.35
        ax.fill_betweenx(y, cx, cx-wl, alpha=0.6, linewidth=0, facecolor=left_color)
        ax.fill_betweenx(y, cx, cx+wr, alpha=0.6, linewidth=0, facecolor=right_color)
        ax.plot([cx, cx], [y.min(), y.max()], linestyle="--", linewidth=1, color=CENTER_COLOR)
        if np.isfinite(lv).any():
            lm = weighted_quantiles(lv, lw, [0.5])[0]
            ax.plot([cx-0.315, cx], [lm, lm], linewidth=2, color=left_color)
        if np.isfinite(rv).any():
            rm = weighted_quantiles(rv, rw, [0.5])[0]
            ax.plot([cx, cx+0.315], [rm, rm], linewidth=2, color=right_color)

    ax.set_xlim(centers[0]-0.6, centers[-1]+0.6)
    ax.set_xticks(centers)
    ax.set_xticklabels(["Low β (Iso)", "Mid β", "High β (Aniso)"],
                       fontweight="bold", fontfamily="DejaVu Sans", color="black")


def row_limits(payloads, keys):
    allv = np.concatenate([payloads[r][k][np.isfinite(payloads[r][k])]
                           for r in REGIONS for k in keys])
    lo, hi = np.nanpercentile(allv, [1, 99])
    pad = 0.07*(hi-lo+1e-12)
    return lo-pad, hi+pad


def stylize_ticks(ax):
    for tick in ax.get_xticklabels():
        tick.set_fontweight("bold")
        tick.set_fontfamily("DejaVu Sans")
        tick.set_color("black")
        tick.set_fontsize(12)
    for tick in ax.get_yticklabels():
        tick.set_fontweight("normal")
        tick.set_fontfamily("DejaVu Sans")
        tick.set_color("black")
        tick.set_fontsize(14)


def build_dashboard(mapping, outpath, multivariate=False):
    payloads = {}
    for reg in REGIONS:
        print(f"\n{reg}:")
        payloads[reg] = prepare_payload(mapping[reg])
    ylim_lai = row_limits(payloads, ["sm_lai", "vpd_lai"])
    ylim_gpp = row_limits(payloads, ["sm_gpp", "vpd_gpp"])

    fig, axes = plt.subplots(2, 5, figsize=(19, 6.3), sharey="row")
    for j, reg in enumerate(REGIONS):
        p = payloads[reg]
        ax0 = axes[0, j]
        l, lw = groups(p["sm_lai"], p["weights"], p["bins"])
        rr, rw = groups(p["vpd_lai"], p["weights"], p["bins"])
        split_violin(ax0, l, rr, lw, rw, SM_LAI, VPD_LAI)
        ax0.set_ylim(*ylim_lai)
        ax0.set_title(REGION_TITLES[reg], fontsize=16, fontweight="bold", color="black")
        ax0.set_xticklabels([])
        stylize_ticks(ax0)
        ax0.grid(True, which="both", linestyle="--", linewidth=0.5, alpha=0.2)

        ax1 = axes[1, j]
        l, lw = groups(p["sm_gpp"], p["weights"], p["bins"])
        rr, rw = groups(p["vpd_gpp"], p["weights"], p["bins"])
        split_violin(ax1, l, rr, lw, rw, SM_GPP, VPD_GPP)
        ax1.set_ylim(*ylim_gpp)
        stylize_ticks(ax1)
        ax1.grid(True, which="both", linestyle="--", linewidth=0.5, alpha=0.2)

    axes[0, 0].set_ylabel("IF strength", fontsize=15, fontweight="bold", color="black")
    axes[1, 0].set_ylabel("IF strength", fontsize=15, fontweight="bold", color="black")
    fig.supxlabel("Isohydricity (β)", y=0.1, fontsize=16, fontweight="bold", color="black")

    if multivariate:
        labels = ["SM→LAI | VPD,T,SSR", "VPD→LAI | SM,T,SSR",
                  "SM→GPP | VPD,T,SSR", "VPD→GPP | SM,T,SSR"]
    else:
        labels = ["SM→LAI", "VPD→LAI", "SM→GPP", "VPD→GPP"]
    handles = [Patch(facecolor=SM_LAI, alpha=0.6, label=labels[0]),
               Patch(facecolor=VPD_LAI, alpha=0.6, label=labels[1]),
               Patch(facecolor=SM_GPP, alpha=0.6, label=labels[2]),
               Patch(facecolor=VPD_GPP, alpha=0.6, label=labels[3])]
    leg = fig.legend(handles=handles, loc="lower center", ncol=4, frameon=False,
                     fontsize=14, bbox_to_anchor=(0.5, 0.012))
    for t in leg.get_texts():
        t.set_fontweight("normal")
        t.set_fontfamily("DejaVu Sans")
        t.set_color("black")
    fig.tight_layout(rect=[0, 0.09, 1, 0.95])
    fig.savefig(outpath, dpi=400, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved {outpath}")


if __name__ == "__main__":
    # Save 
    out_bi = BASE / "IF_beta_bivariate.png"
    out_mv = BASE / "IF_beta_multivariate.png"
    build_dashboard(bivar, out_bi, multivariate=False)
    build_dashboard(multivar, out_mv, multivariate=True)
